# Linear Algebra for Quantum

Five concepts, in code first, then in notation.

**Objectives:**
- Compute inner products and use them to test orthogonality
- Check that a matrix is **unitary** (this is the defining property of every quantum gate)
- Take conjugate transposes (the dagger `†`)
- Build multi-qubit states with tensor products

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. Inner product

The **inner product** of two complex vectors `a` and `b` is

$$\langle a, b \rangle = \sum_i \bar{a_i} \, b_i$$

In NumPy that is `a.conj() @ b`. It produces a single complex number that measures how much `a` and `b` overlap.

In [ ]:
a = np.array([1, 0])           # the 'x-axis' in 2D
b = np.array([0, 1])           # the 'y-axis'
c = np.array([1, 1]) / np.sqrt(2)

print('<a, b> =', a.conj() @ b)   # 0  — orthogonal
print('<a, c> =', a.conj() @ c)   # 1/sqrt(2)
print('<c, c> =', c.conj() @ c)   # 1  — c is a unit vector

**Important property:** `<v, v>` equals the squared norm of `v`. Quantum states are *always* unit vectors: `<psi, psi> = 1`.

## 2. Conjugate transpose (the dagger)

The **conjugate transpose** of a matrix `M` is written `M†` and obtained by transposing AND conjugating. In NumPy: `M.conj().T`.

In [ ]:
M = np.array([
    [1+1j, 2],
    [3,    4-1j],
])
print('M =')
print(M)
print('\nM† =')
print(M.conj().T)

## 3. Unitary matrices

A matrix `U` is **unitary** if

$$U^\dagger U = I$$

That is the entire definition. Every quantum gate is unitary, because unitary matrices preserve the norm of a state — and quantum states must stay unit vectors.

Let's verify it for two famous matrices.

In [ ]:
I = np.eye(2)

# The Hadamard gate
H = (1 / np.sqrt(2)) * np.array([
    [1,  1],
    [1, -1],
])
print('H† H =')
print(H.conj().T @ H)
print('Is H unitary?', np.allclose(H.conj().T @ H, I))

# The Pauli-Y gate
Y = np.array([[0, -1j], [1j, 0]])
print('\nY† Y =')
print(Y.conj().T @ Y)
print('Is Y unitary?', np.allclose(Y.conj().T @ Y, I))

In [ ]:
# A handy reusable helper.
def is_unitary(U, tol=1e-10):
    U = np.asarray(U, dtype=complex)
    n = U.shape[0]
    return np.allclose(U.conj().T @ U, np.eye(n), atol=tol)

print(is_unitary(H), is_unitary(Y), is_unitary(np.array([[1, 2], [3, 4]])))

## 4. Tensor products build multi-qubit states

If qubit A is in state `a` (a 2-vector) and qubit B is in state `b` (a 2-vector), then the joint state of the two qubits is the **tensor product**

$$|a\rangle \otimes |b\rangle$$

which is a 4-vector. In NumPy: `np.kron(a, b)`.

For `n` qubits, the joint state is a `2^n`-vector. This exponential blow-up is what makes classical simulation hard — and what gives quantum computers their potential leverage.

In [ ]:
zero = np.array([1, 0])
one  = np.array([0, 1])
plus = (zero + one) / np.sqrt(2)

# Two qubits, both in |0>
psi_00 = np.kron(zero, zero)
print('|00> =', psi_00)

# First qubit in |+>, second in |0>
psi = np.kron(plus, zero)
print('|+>|0> =', psi)
print('norm:', np.linalg.norm(psi))    # always 1

## 5. Notation cheat sheet (commit to memory)

| Math | NumPy | Meaning |
|---|---|---|
| $\langle a \| b \rangle$ | `a.conj() @ b` | Inner product, returns a scalar |
| $\|v\|$ | `np.linalg.norm(v)` | Length of `v` |
| $M^\dagger$ | `M.conj().T` | Conjugate transpose |
| $U^\dagger U = I$ | `np.allclose(U.conj().T @ U, I)` | Unitarity test |
| $A \otimes B$ | `np.kron(A, B)` | Tensor product |


## 6. Exercises

Three exercises to lock in the notation. Each has tiered hints -- expand
only what you need -- and a check cell that tells you when you have it.
Worked solutions wait at the bottom; run the checks before peeking.

### Exercise 1 — Orthogonal unit vectors

Verify with inner products that the computational basis states $|0\rangle$
and $|1\rangle$ are orthogonal, and that each is a unit vector.

Define `ket0` and `ket1` — the two basis vectors as NumPy arrays — and the
three inner products `ip_01` ($\langle 0|1\rangle$), `ip_00`
($\langle 0|0\rangle$), and `ip_11` ($\langle 1|1\rangle$).

<details><summary>Hint 1 — nudge</summary>

One tool from Section 1 answers both questions: orthogonality means the
inner product of the two *different* vectors is 0, and the inner product of
a vector with *itself* is its squared norm — 1 for a unit vector.

</details>
<details><summary>Hint 2 — approach</summary>

Build the two 2-entry arrays with `np.array`, then form each inner product
with the `a.conj() @ b` pattern from Section 1 — the mixed pair once, and
each vector with itself.

</details>


In [ ]:
# Exercise 1: Verify |0> and |1> are orthogonal unit vectors.
# Define: ket0, ket1 -- the basis vectors -- and the inner products
# ip_01, ip_00, ip_11.

# TODO: your code here


In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(ket0) == 2 and len(ket1) == 2, (
        "single-qubit basis vectors have exactly 2 entries"
    )
    assert np.isclose(ip_01, 0), (
        "orthogonal vectors have inner product 0 -- recheck <0|1>"
    )
    assert np.isclose(ip_00, 1) and np.isclose(ip_11, 1), (
        "a unit vector's inner product with itself is 1"
    )


### Exercise 2 — Unitary or not?

Section 3 gave the whole definition: a matrix is unitary exactly when
$U^\dagger U = I$. Decide whether

$$A = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$$

is unitary — by computing, not by eyeballing.

Define `A_matrix` (the matrix above), `AdagA` (the product $A^\dagger A$),
and `A_is_unitary` (`True` or `False`, confirmed with `is_unitary`).

<details><summary>Hint 1 — nudge</summary>

A unitary matrix has orthonormal columns: $U^\dagger U = I$ says every
column has norm 1 and distinct columns are orthogonal. Look at the two
columns of $A$ with that in mind.

</details>
<details><summary>Hint 2 — approach</summary>

Build the matrix with `np.array`, form the dagger with the `.conj().T`
pattern from Section 2, multiply with `@`, and compare the result against
`np.eye(2)`. The `is_unitary` helper from Section 3 should agree with what
you see.

</details>


In [ ]:
# Exercise 2: Is A = [[1, 1], [1, 1]] unitary?
# Define: A_matrix, AdagA -- the product A-dagger A -- and A_is_unitary
# (True or False).

# TODO: your code here


In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert np.allclose(A_matrix, [[1, 1], [1, 1]]), (
        "build A exactly as given in the prompt"
    )
    assert np.asarray(AdagA).shape == (2, 2), (
        "A-dagger A of a 2x2 matrix is again 2x2"
    )
    assert np.allclose(AdagA, np.asarray(AdagA).conj().T), (
        "any M-dagger M is Hermitian -- if yours is not, recheck the product order"
    )
    assert bool(A_is_unitary) == bool(np.allclose(AdagA, np.eye(2))), (
        "your verdict should match the U-dagger U = I test applied to your product"
    )
    assert not A_is_unitary, (
        "compare each column's norm with 1 -- unitary matrices have orthonormal columns"
    )


### Exercise 3 — A three-qubit product state

Build the 3-qubit state $|0\rangle \otimes |1\rangle \otimes |0\rangle$
and locate its single nonzero entry.

Define `psi_010` — the full state vector — and `hot_index` — the integer
index of the entry that equals 1.

<details><summary>Hint 1 — nudge</summary>

`np.kron` combines two vectors at a time, and each application doubles the
length — Section 4 built two-qubit states this way. How many applications
do three qubits take, and how long must the result be?

</details>
<details><summary>Hint 2 — approach</summary>

Nest the calls: tensor the first two single-qubit vectors, then tensor that
result with the third. To locate the nonzero entry, `np.argmax` over the
absolute values gives its index — and reading that index in binary should
spell the qubit pattern.

</details>


In [ ]:
# Exercise 3: Build |0> (x) |1> (x) |0> and locate its nonzero entry.
# Define: psi_010 -- the 8-entry state vector -- and hot_index (int).

# TODO: your code here


In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert len(psi_010) == 8, (
        "three qubits live in a 2^3 = 8 dimensional space"
    )
    assert np.isclose(np.linalg.norm(psi_010), 1), (
        "a tensor product of unit vectors is still a unit vector"
    )
    assert np.count_nonzero(np.abs(np.asarray(psi_010)) > 1e-9) == 1, (
        "a product of basis states is a basis state -- exactly one entry is nonzero"
    )
    assert np.isclose(np.abs(np.asarray(psi_010)[int(hot_index)]), 1), (
        "hot_index should point at the entry that equals 1"
    )
    assert format(int(hot_index), "03b") == "010", (
        "read the index in binary -- it should spell the qubit pattern"
    )


### Solutions

In [ ]:
# --- Exercise 1 ---
ket0 = np.array([1, 0])
ket1 = np.array([0, 1])
ip_01 = ket0.conj() @ ket1
ip_00 = ket0.conj() @ ket0
ip_11 = ket1.conj() @ ket1
print('<0|1> =', ip_01)                   # 0 -- orthogonal
print('<0|0> =', ip_00)                   # 1 -- unit vector
print('<1|1> =', ip_11)                   # 1 -- unit vector

# --- Exercise 2 ---
A_matrix = np.array([[1, 1], [1, 1]])
AdagA = A_matrix.conj().T @ A_matrix
A_is_unitary = is_unitary(A_matrix)
print('A† A =')
print(AdagA)                              # [[2, 2], [2, 2]] -- not the identity
print('Is A unitary?', A_is_unitary)      # False -- its columns aren't orthonormal

# --- Exercise 3 ---
psi_010 = np.kron(np.kron(zero, one), zero)
hot_index = int(np.argmax(np.abs(psi_010)))
print('length:', len(psi_010))            # 8
print('nonzero index:', hot_index)        # 2  (binary 010)

**You finished notebook 2.** Move on to [`03-probability-and-measurement.ipynb`](03-probability-and-measurement.ipynb).